In [ ]:
!pip install Faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from faker import Faker
import random
from datetime import datetime, timedelta

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Using device: cpu


In [ ]:
fake = Faker('en_IN')  # Use Indian locale for realistic names, addresses

# Hardcoded data from provided documents
# States from Census XLSX
states = [
    'A & N Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh',
    'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir',
    'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya',
    'Mizoram', 'Nagaland', 'Odisha', 'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu',
    'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal'
]

# Worker categories from DDWCT XLS (simplified: Main workers, Marginal, Non-workers)
worker_types = ['Main workers', 'Marginal workers', 'Non-workers']

# Age groups from DDWCT
age_groups = ['5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-49', '50-59', '60-69', '70-79', '80+']

# Economic data: Sample GSDP from Census XLSX (2011-12 base, in Lakh, for manufacturing)
# Using approximate values for states (from row6 to row38)
gsdp_manufacturing = {
    'A & N Islands': 5051,
    'Andhra Pradesh': 3969153,
    'Arunachal Pradesh': 9934,
    'Assam': 1274688,
    'Bihar': 1268069,
    'Chandigarh': 91508,
    'Chhattisgarh': 2096873,
    'Delhi': 1591118,
    'Goa': 1508357,
    'Gujarat': 11184923,
    'Haryana': 4454729,
    'Himachal Pradesh': 1393182,
    'Jammu and Kashmir': 624271,
    'Jharkhand': 2740497,
    'Karnataka': 8343501,
    'Kerala': 3000616,
    'Madhya Pradesh': 2967896,
    'Maharashtra': 20680408,
    'Manipur': 32858,
    'Meghalaya': 426022,
    'Mizoram': 4967,
    'Nagaland': 11152,
    'Odisha': 3273608,
    'Puducherry': 368161,
    'Punjab': 2940858,
    'Rajasthan': 5689422,
    'Sikkim': 413379,
    'Tamil Nadu': 12163483,
    'Telangana': 5292515,
    'Tripura': 62462,
    'Uttar Pradesh': 6919442,
    'Uttarakhand': 3741191,
    'West Bengal': 0  # Not available, set to 0
}

# RBI regulations compliance: Simulate privacy by anonymizing, use synthetic IDs, ensure no sensitive real data.
# Incorporate NSFR-like stability: Assign random RSF factors (e.g., 65% for certain loans as per regs).
# Inoperative accounts: Randomly flag some as inoperative.
# Hedging: Simulate FX risk hedging flags.
# Dates post-2023 to align with recent regs.

def generate_synthetic_applications(num_rows=1000, is_train=True):
    data = []
    for _ in range(num_rows):
        state = random.choice(states)
        age_group = random.choice(age_groups)
        age_min, age_max = map(int, age_group.split('-')) if '-' in age_group else (int(age_group[:-1]), 100)
        age = random.randint(age_min, age_max)
        gender = random.choice(['Males', 'Females'])
        worker_type = random.choice(worker_types)
        income = np.random.normal(gsdp_manufacturing[state] / 1000, 100000)  # Scale to personal income, synthetic
        income = max(10000, abs(income))  # Minimum income
        loan_amount = random.randint(50000, 5000000)  # Synthetic loan amounts
        target = random.choice([0, 1]) if is_train else None  # Default or not

        row = {
            'SK_ID_CURR': fake.unique.random_int(min=100000, max=999999),
            'TARGET': target,
            'NAME_CONTRACT_TYPE': random.choice(['Cash loans', 'Revolving loans']),
            'CODE_GENDER': 'M' if gender == 'Males' else 'F',
            'FLAG_OWN_CAR': random.choice(['Y', 'N']),
            'FLAG_OWN_REALTY': random.choice(['Y', 'N']),
            'CNT_CHILDREN': random.randint(0, 5),
            'AMT_INCOME_TOTAL': income,
            'AMT_CREDIT': loan_amount,
            'AMT_ANNUITY': loan_amount * 0.1,  # Synthetic
            'NAME_INCOME_TYPE': worker_type,
            'REGION_POPULATION_RELATIVE': random.uniform(0.01, 0.05),
            'DAYS_BIRTH': -age * 365,  # Negative days
            'DAYS_EMPLOYED': random.randint(-10000, -100) if worker_type != 'Non-workers' else 0,
            'OWN_CAR_AGE': random.randint(1, 20) if random.choice([True, False]) else np.nan,
            # RBI compliance: Flag for NSFR (synthetic)
            'NSFR_RSF_FACTOR': 65 if random.random() > 0.5 else 100,  # As per Dec 29, 2023 reg
            'INOPERATIVE_FLAG': random.choice([0, 1]),  # Inoperative accounts reg
            'STATE': state,
            # Add more columns as needed, total ~122 in real dataset, but simplified
        }
        data.append(row)

    df = pd.DataFrame(data)
    filename = 'application_train.csv' if is_train else 'application_test.csv'
    df.to_csv(filename, index=False)
    return filename

def generate_synthetic_bureau(num_rows=500):
    data = []
    for _ in range(num_rows):
        row = {
            'SK_ID_CURR': fake.random_int(min=100000, max=999999),
            'SK_ID_BUREAU': fake.unique.random_int(min=5000000, max=5999999),
            'CREDIT_ACTIVE': random.choice(['Active', 'Closed', 'Sold']),
            'CREDIT_CURRENCY': 'INR',  # Indian context
            'DAYS_CREDIT': -random.randint(100, 3000),
            'CREDIT_DAY_OVERDUE': random.randint(0, 100),
            'DAYS_CREDIT_ENDDATE': random.randint(-1000, 1000),
            'AMT_CREDIT_MAX_OVERDUE': random.uniform(0, 100000),
            'CNT_CREDIT_PROLONG': random.randint(0, 5),
            'AMT_CREDIT_SUM': random.uniform(50000, 5000000),
            'AMT_CREDIT_SUM_DEBT': random.uniform(0, 3000000),
            'AMT_CREDIT_SUM_LIMIT': random.uniform(0, 1000000),
            'AMT_CREDIT_SUM_OVERDUE': random.uniform(0, 500000),
            'CREDIT_TYPE': random.choice(['Consumer credit', 'Credit card', 'Mortgage', 'Car loan']),
            'DAYS_CREDIT_UPDATE': -random.randint(1, 100),
            'AMT_ANNUITY': random.uniform(1000, 100000),
            # RBI: Hedging flag for FX risk (Jan 5, 2024)
            'FX_HEDGING_FLAG': random.choice([0, 1]),
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv('bureau.csv', index=False)
    return 'bureau.csv'

# Similarly, define functions for other files (simplified)
def generate_synthetic_bureau_balance(num_rows=1000):
    data = []
    for _ in range(num_rows):
        row = {
            'SK_ID_BUREAU': fake.random_int(min=5000000, max=5999999),
            'MONTHS_BALANCE': -random.randint(0, 100),
            'STATUS': random.choice(['0', '1', '2', '3', '4', '5', 'C', 'X']),
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv('bureau_balance.csv', index=False)
    return 'bureau_balance.csv'

def generate_synthetic_pos_cash_balance(num_rows=1000):
    data = []
    for _ in range(num_rows):
        row = {
            'SK_ID_PREV': fake.random_int(min=1000000, max=1999999),
            'SK_ID_CURR': fake.random_int(min=100000, max=999999),
            'MONTHS_BALANCE': -random.randint(1, 100),
            'CNT_INSTALMENT': random.randint(1, 48),
            'CNT_INSTALMENT_FUTURE': random.randint(0, 48),
            'NAME_CONTRACT_STATUS': random.choice(['Active', 'Completed', 'Demand', 'Signed']),
            'SK_DPD': random.randint(0, 10),
            'SK_DPD_DEF': random.randint(0, 5),
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv('POS_CASH_balance.csv', index=False)
    return 'POS_CASH_balance.csv'

def generate_synthetic_credit_card_balance(num_rows=1000):
    data = []
    for _ in range(num_rows):
        row = {
            'SK_ID_PREV': fake.random_int(min=1000000, max=1999999),
            'SK_ID_CURR': fake.random_int(min=100000, max=999999),
            'MONTHS_BALANCE': -random.randint(1, 100),
            'AMT_BALANCE': random.uniform(-10000, 1000000),
            'AMT_CREDIT_LIMIT_ACTUAL': random.randint(10000, 1000000),
            'AMT_DRAWINGS_ATM_CURRENT': random.uniform(0, 500000),
            # ... add more
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv('credit_card_balance.csv', index=False)
    return 'credit_card_balance.csv'

def generate_synthetic_previous_application(num_rows=500):
    data = []
    for _ in range(num_rows):
        app_date = fake.date_between(start_date='-5y', end_date='today')
        row = {
            'SK_ID_PREV': fake.unique.random_int(min=1000000, max=1999999),
            'SK_ID_CURR': fake.random_int(min=100000, max=999999),
            'NAME_CONTRACT_TYPE': random.choice(['Cash loans', 'Consumer loans', 'Revolving loans']),
            'AMT_ANNUITY': random.uniform(1000, 100000),
            'AMT_APPLICATION': random.uniform(50000, 5000000),
            'AMT_CREDIT': random.uniform(50000, 5000000),
            'AMT_DOWN_PAYMENT': random.uniform(0, 1000000),
            'AMT_GOODS_PRICE': random.uniform(50000, 5000000),
            'WEEKDAY_APPR_PROCESS_START': random.choice(['MONDAY', 'TUESDAY', 'WEDNESDAY', 'THURSDAY', 'FRIDAY', 'SATURDAY', 'SUNDAY']),
            'HOUR_APPR_PROCESS_START': random.randint(0, 23),
            'FLAG_LAST_APPL_PER_CONTRACT': 'Y',
            'NFLAG_LAST_APPL_IN_DAY': 1,
            'RATE_DOWN_PAYMENT': random.uniform(0, 0.5),
            'RATE_INTEREST_PRIMARY': random.uniform(0.05, 0.3),  # RBI regulated rates
            # RBI: Commercial Paper flag (Jan 3, 2024)
            'CP_NCD_FLAG': random.choice([0, 1]),
            'NAME_CASH_LOAN_PURPOSE': random.choice(['XAP', 'XNA']),
            # ... add more
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv('previous_application.csv', index=False)
    return 'previous_application.csv'

def generate_synthetic_installments_payments(num_rows=2000):
    data = []
    for _ in range(num_rows):
        row = {
            'SK_ID_PREV': fake.random_int(min=1000000, max=1999999),
            'SK_ID_CURR': fake.random_int(min=100000, max=999999),
            'NUM_INSTALMENT_VERSION': random.uniform(0, 2),
            'NUM_INSTALMENT_NUMBER': random.randint(1, 100),
            'DAYS_INSTALMENT': -random.randint(1, 3000),
            'DAYS_ENTRY_PAYMENT': -random.randint(1, 3000),
            'AMT_INSTALMENT': random.uniform(1000, 100000),
            'AMT_PAYMENT': random.uniform(1000, 100000),
        }
        data.append(row)

    df = pd.DataFrame(data)
    df.to_csv('installments_payments.csv', index=False)
    return 'installments_payments.csv'

# Generate descriptions (static)
def generate_columns_description():
    # Simplified, in real it's a CSV with descriptions
    data = [
        {'TableName': 'application_{train|test}', 'Column': 'SK_ID_CURR', 'Description': 'ID of loan in our sample'},
        # Add more for all columns...
    ]
    df = pd.DataFrame(data)
    df.to_csv('HomeCredit_columns_description.csv', index=False)
    return 'HomeCredit_columns_description.csv'

# Main function to generate all
def generate_all_synthetic_data(num_apps=1000):
    print("Generating application_train.csv")
    generate_synthetic_applications(num_apps, is_train=True)
    print("Generating application_test.csv")
    generate_synthetic_applications(num_apps // 2, is_train=False)
    print("Generating bureau.csv")
    generate_synthetic_bureau(num_apps * 2)
    print("Generating bureau_balance.csv")
    generate_synthetic_bureau_balance(num_apps * 10)
    print("Generating POS_CASH_balance.csv")
    generate_synthetic_pos_cash_balance(num_apps * 10)
    print("Generating credit_card_balance.csv")
    generate_synthetic_credit_card_balance(num_apps * 10)
    print("Generating previous_application.csv")
    generate_synthetic_previous_application(num_apps * 2)
    print("Generating installments_payments.csv")
    generate_synthetic_installments_payments(num_apps * 20)
    print("Generating HomeCredit_columns_description.csv")
    generate_columns_description()
    print("All synthetic data files generated.")

# Run
generate_all_synthetic_data(100)

Generating application_train.csv
Generating application_test.csv
Generating bureau.csv
Generating bureau_balance.csv
Generating POS_CASH_balance.csv
Generating credit_card_balance.csv
Generating previous_application.csv
Generating installments_payments.csv
Generating HomeCredit_columns_description.csv
All synthetic data files generated.


In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from scipy import stats
from collections import defaultdict
import os
import warnings
warnings.filterwarnings("ignore")

fake = Faker('en_IN')

# --- RBI Compliance Flags (Hardcoded for Regulation) ---
RBI_FLAGS = {
    'NSFR_RSF_FACTOR': [65, 100],           # Dec 29, 2023: NDB NSFR
    'INOPERATIVE_FLAG': [0, 1],             # Jan 01, 2024: Inoperative accounts
    'FX_HEDGING_FLAG': [0, 1],              # Jan 05, 2024: FX risk hedging
    'CP_NCD_FLAG': [0, 1],                  # Jan 03, 2024: CP/NCD consistency
}

# --- Census Anchors (Optional: fallback if no input data) ---
CENSUS_STATES = [
    'A & N Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh',
    'Chhattisgarh', 'Delhi', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh',
    'Jammu and Kashmir', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh',
    'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha',
    'Puducherry', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana',
    'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal'
]

# --- Distribution Learners ---
class DistributionSampler:
    def __init__(self):
        self.samplers = {}
        self.correlations = {}

    def fit(self, df, col):
        if col not in df.columns:
            return
        series = df[col].dropna()
        if series.empty:
            return

        if series.dtype == 'object' or series.dtype.name == 'category':
            values, probs = np.unique(series, return_counts=True)
            probs = probs / probs.sum()
            self.samplers[col] = lambda size=1: np.random.choice(values, size=size, p=probs)
        elif np.issubdtype(series.dtype, np.number):
            if series.nunique() < 10:
                # Treat as discrete
                values, probs = np.unique(series, return_counts=True)
                probs = probs / probs.sum()
                self.samplers[col] = lambda size=1: np.random.choice(values, size=size, p=probs)
            else:
                mu, sigma = stats.norm.fit(series)
                self.samplers[col] = lambda size=1: stats.truncnorm.rvs(
                    (0 - mu) / sigma, (np.inf - mu) / sigma, loc=mu, scale=sigma, size=size)
        elif 'DATE' in str(series.dtype).upper() or 'date' in str(series.dtype):
            series_dt = pd.to_datetime(series, errors='coerce').dropna()
            if not series_dt.empty:
                min_date, max_date = series_dt.min(), series_dt.max()
                days_range = (max_date - min_date).days
                self.samplers[col] = lambda size=1: [
                    (min_date + pd.Timedelta(days=random.randint(0, days_range))).date()
                    for _ in range(size[0] if hasattr(size, '__len__') else size)
                ]

    def sample(self, col, size=1):
        if col in self.samplers:
            return self.samplers[col](size)
        else:
            return [None] * size if size > 1 else None

# --- Multi-Dataset Synthetic Generator ---
def generate_synthetic_from_datasets(
    input_files_dict,
    output_size_multiplier=1.0,
    seed=42
):
    """
    input_files_dict = {
        'application': ['app1.csv', 'app2.csv'],
        'bureau': ['bureau1.csv'],
        'previous_application': ['prev1.csv', 'prev2.csv'],
        # ... add others as needed
    }
    """
    random.seed(seed)
    np.random.seed(seed)
    Faker.seed(seed)

    # --- Step 1: Load and Combine Input Data ---
    combined = {}
    for table_name, files in input_files_dict.items():
        if not files:
            continue
        dfs = []
        for f in files:
            if os.path.exists(f):
                df = pd.read_csv(f, low_memory=False)
                dfs.append(df)
        if dfs:
            combined[table_name] = pd.concat(dfs, ignore_index=True)
        else:
            combined[table_name] = None

    # --- Step 2: Fit Samplers per Table ---
    samplers = defaultdict(DistributionSampler)
    key_columns = {
        'application': 'SK_ID_CURR',
        'bureau': 'SK_ID_CURR',
        'previous_application': 'SK_ID_CURR',
        'bureau_balance': 'SK_ID_BUREAU',
        'POS_CASH_balance': 'SK_ID_CURR',
        'credit_card_balance': 'SK_ID_CURR',
        'installments_payments': 'SK_ID_CURR'
    }

    for table, df in combined.items():
        if df is not None:
            for col in df.columns:
                if col not in ['SK_ID_CURR', 'SK_ID_BUREAU', 'SK_ID_PREV', 'TARGET']:
                    samplers[table].fit(df, col)

    # --- Step 3: Generate Synthetic Data ---
    output_files = {}

    # --- Application (Train + Test) ---
    app_df = combined.get('application')
    if app_df is not None:
        n_rows = int(len(app_df) * output_size_multiplier)
        app_sampler = samplers['application']

        # Train set
        train_data = []
        for _ in range(n_rows):
            row = {col: app_sampler.sample(col)[0] for col in app_df.columns if col != 'TARGET'}
            row['SK_ID_CURR'] = fake.unique.random_int(100000, 999999)
            row['TARGET'] = int(np.random.rand() < app_df['TARGET'].mean()) if 'TARGET' in app_df.columns else None
            # Inject RBI flags
            row['NSFR_RSF_FACTOR'] = random.choice(RBI_FLAGS['NSFR_RSF_FACTOR'])
            row['INOPERATIVE_FLAG'] = 1 if random.random() < 0.03 else 0
            row['STATE'] = row.get('STATE', random.choice(CENSUS_STATES))
            train_data.append(row)

        train_df = pd.DataFrame(train_data)
        train_file = 'application_train.csv'
        train_df.to_csv(train_file, index=False)
        output_files['application_train'] = train_file

        # Test set (no TARGET)
        test_df = train_df.drop(columns=['TARGET'], errors='ignore')
        test_file = 'application_test.csv'
        test_df.to_csv(test_file, index=False)
        output_files['application_test'] = test_file

    # --- Bureau ---
    bureau_df = combined.get('bureau')
    if bureau_df is not None:
        n_bureau = int(len(bureau_df) * output_size_multiplier)
        bureau_sampler = samplers['bureau']
        bureau_data = []
        for _ in range(n_bureau):
            row = {col: bureau_sampler.sample(col)[0] for col in bureau_df.columns if col not in ['SK_ID_CURR', 'SK_ID_BUREAU']}
            row['SK_ID_BUREAU'] = fake.unique.random_int(5000000, 5999999)
            row['SK_ID_CURR'] = random.choice(train_df['SK_ID_CURR']) if 'train_df' in locals() else fake.random_int(100000, 999999)
            row['FX_HEDGING_FLAG'] = random.choice(RBI_FLAGS['FX_HEDGING_FLAG'])
            bureau_data.append(row)
        bureau_final = pd.DataFrame(bureau_data)
        bureau_file = 'bureau.csv'
        bureau_final.to_csv(bureau_file, index=False)
        output_files['bureau'] = bureau_file

    # --- Previous Application ---
    prev_df = combined.get('previous_application')
    if prev_df is not None:
        n_prev = int(len(prev_df) * output_size_multiplier)
        prev_sampler = samplers['previous_application']
        prev_data = []
        for _ in range(n_prev):
            row = {col: prev_sampler.sample(col)[0] for col in prev_df.columns if col not in ['SK_ID_CURR', 'SK_ID_PREV']}
            row['SK_ID_PREV'] = fake.unique.random_int(1000000, 1999999)
            row['SK_ID_CURR'] = random.choice(train_df['SK_ID_CURR']) if 'train_df' in locals() else fake.random_int(100000, 999999)
            row['CP_NCD_FLAG'] = random.choice(RBI_FLAGS['CP_NCD_FLAG'])
            prev_data.append(row)
        prev_final = pd.DataFrame(prev_data)
        prev_file = 'previous_application.csv'
        prev_final.to_csv(prev_file, index=False)
        output_files['previous_application'] = prev_file

    # --- Other Tables (Simplified: Use same logic) ---
    # POS_CASH_balance, credit_card_balance, installments_payments, bureau_balance
    # (Can be extended similarly — omitted for brevity)

    # --- Column Description ---
    desc_data = [
        {'Table': 'application_train', 'Column': 'SK_ID_CURR', 'Description': 'Synthetic ID of loan application'},
        {'Table': 'application_train', 'Column': 'TARGET', 'Description': 'Synthetic default indicator (1 = default)'},
        {'Table': 'application_train', 'Column': 'NSFR_RSF_FACTOR', 'Description': 'RBI NSFR RSF factor (65% or 100%)'},
        {'Table': 'application_train', 'Column': 'INOPERATIVE_FLAG', 'Description': 'RBI inoperative account flag'},
        # Add more...
    ]
    desc_df = pd.DataFrame(desc_data)
    desc_file = 'HomeCredit_columns_description.csv'
    desc_df.to_csv(desc_file, index=False)
    output_files['description'] = desc_file

    print("Synthetic data generation complete:")
    for name, path in output_files.items():
        print(f"  - {path}")

    return output_files

# === USAGE EXAMPLE ===
"""
input_data = {
    'application': ['app_train_sample1.csv', 'app_train_sample2.csv'],
    'bureau': ['bureau_sample.csv'],
    'previous_application': ['previous_app_sample.csv']
}

generate_synthetic_from_datasets(input_data, output_size_multiplier=1.5)
"""